# Лабораторна робота №5: стиснення зображень

Досліджуємо **стиснення з втратами** на основі **дискретного косинусного перетворення (DCT)** по блоках \(8 \times 8\), **квантування** коефіцієнтів за матрицею як у JPEG (яскравісний канал) та **оцінку якості** відновленого зображення (MSE, PSNR). Вхідне зображення — `satir.jpg` у градаціях сірого; результати — у `Lab_05/results/`.

## 1. Імпорт бібліотек

Використовуємо `pathlib`, `numpy`, OpenCV (`cv2`), `matplotlib` для графіків, а також **`scipy.fftpack.dct`** та **`scipy.fftpack.idct`** для 1D DCT/IDCT у роздільному обчисленні 2D DCT по рядках і стовпцях.

In [1]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from scipy.fftpack import dct, idct

## 2. Налаштування шляхів

Якщо поточна папка містить `Lab_05.ipynb`, вважаємо, що Jupyter запущено з `Lab_05`; інакше беремо підпапку `Lab_05` від кореня репозиторію. Тоді `ROOT` — батько `NOTEBOOK_DIR`, а `satir.jpg` лежить у `ROOT`.

In [2]:
_cwd = Path.cwd().resolve()
NOTEBOOK_DIR = _cwd if (_cwd / "Lab_05.ipynb").is_file() else (_cwd / "Lab_05")
ROOT = NOTEBOOK_DIR.parent.resolve()
IMAGE_PATH = ROOT / "satir.jpg"
RESULTS_DIR = (NOTEBOOK_DIR / "results").resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("ROOT:", ROOT)
print("IMAGE_PATH:", IMAGE_PATH)
print("RESULTS_DIR:", RESULTS_DIR)

NOTEBOOK_DIR: C:\Users\feost\OneDrive\Документи\CNU Education\AISIP\Lab_05
ROOT: C:\Users\feost\OneDrive\Документи\CNU Education\AISIP
IMAGE_PATH: C:\Users\feost\OneDrive\Документи\CNU Education\AISIP\satir.jpg
RESULTS_DIR: C:\Users\feost\OneDrive\Документи\CNU Education\AISIP\Lab_05\results


## 3. Читання / запис з Unicode-шляхами (Windows)

Обхід обмежень `cv2.imread` / `imwrite` для не-ASCII шляхів через `numpy.fromfile` + `cv2.imdecode` та `imencode` + `tofile`.

In [3]:
def imread_gray_unicode(path: Path) -> np.ndarray:
    data = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Не вдалося прочитати: {path}")
    return img


def imwrite_unicode(path: Path, image: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    ok, buf = cv2.imencode(path.suffix, image)
    if not ok:
        raise RuntimeError(f"Не вдалося закодувати: {path}")
    buf.tofile(str(path))

## 4. Завантаження зображення

Зберігаємо копію вихідного сірого зображення для подальшого порівняння.

In [4]:
img_gray = imread_gray_unicode(IMAGE_PATH)
imwrite_unicode(RESULTS_DIR / "original_gray.png", img_gray)
h, w = img_gray.shape
print(f"Розмір: {w}×{h} пікселів")

Розмір: 426×640 пікселів


## 5. DCT та IDCT для блоків \(8 \times 8\)

Двовимірне DCT-II з ортонормованим нормуванням реалізуємо через послідовне застосування одновимірного DCT по стовпцях і рядках (як у завданні). Зворотне перетворення — через `idct` з тим самим `norm="ortho"`.

In [5]:
def dct2(block: np.ndarray) -> np.ndarray:
    return dct(dct(block.T, norm="ortho").T, norm="ortho")


def idct2(block: np.ndarray) -> np.ndarray:
    return idct(idct(block.T, norm="ortho").T, norm="ortho")

## 6. Квантування та відновлення

Використовуємо **стандартну матрицю квантування яскравості JPEG** (як для сірого каналу). Параметр якості \(Q \in [1, 100]\) масштабує матрицю за типовою схемою ISO: при меншому \(Q\) крок квантування більший — більше втрат і вища ступінь «стиснення» у сенсі спрощення коефіцієнтів.

Для кожного блоку: зсув на \(-128\), `dct2`, ділення на матрицю квантування, округлення, зворотне множення, `idct2`, зсув на \(+128\). Зображення доповнюємо до кратного 8 по краю (`edge`), після відновлення обрізаємо до початкового розміру.

In [6]:
Q_JPEG_LUMA = np.array(
    [
        [16, 11, 10, 16, 24, 40, 51, 61],
        [12, 12, 14, 19, 26, 58, 60, 55],
        [14, 13, 16, 24, 40, 57, 69, 56],
        [14, 17, 22, 29, 51, 87, 80, 62],
        [18, 22, 37, 56, 68, 109, 103, 77],
        [24, 35, 55, 64, 81, 104, 113, 92],
        [49, 64, 78, 87, 103, 121, 120, 101],
        [72, 92, 95, 98, 112, 100, 103, 99],
    ],
    dtype=np.float64,
)


def quantization_matrix(quality: int) -> np.ndarray:
    q = int(np.clip(quality, 1, 100))
    if q < 50:
        scale = 5000.0 / q
    else:
        scale = 200.0 - 2.0 * q
    mat = np.floor((Q_JPEG_LUMA * scale + 50.0) / 100.0)
    return np.maximum(mat, 1.0)


def compress_decompress_block(block: np.ndarray, Qmat: np.ndarray) -> np.ndarray:
    x = block.astype(np.float64) - 128.0
    c = dct2(x)
    cq = np.round(c / Qmat)
    r = idct2(cq * Qmat) + 128.0
    return r


def compress_decompress_image(gray: np.ndarray, quality: int) -> np.ndarray:
    Qm = quantization_matrix(quality)
    hh, ww = gray.shape
    pad_h = (8 - hh % 8) % 8
    pad_w = (8 - ww % 8) % 8
    padded = np.pad(gray, ((0, pad_h), (0, pad_w)), mode="edge")
    H, W = padded.shape
    out = np.zeros_like(padded, dtype=np.float64)
    for i in range(0, H, 8):
        for j in range(0, W, 8):
            blk = padded[i : i + 8, j : j + 8]
            out[i : i + 8, j : j + 8] = compress_decompress_block(blk, Qm)
    out = np.clip(np.round(out), 0, 255).astype(np.uint8)
    return out[:hh, :ww]


def mse_psnr(original: np.ndarray, restored: np.ndarray) -> tuple[float, float]:
    a = original.astype(np.float64)
    b = restored.astype(np.float64)
    mse = float(np.mean((a - b) ** 2))
    if mse == 0.0:
        return mse, float("inf")
    psnr = 10.0 * np.log10(255.0**2 / mse)
    return mse, float(psnr)

## 7. Експеримент: різні рівні якості

Будуємо відновлені зображення для кількох значень \(Q\), обчислюємо **MSE** та **PSNR**, зберігаємо окремі PNG та візуалізацію спектру DCT для першого блоку (наочність коефіцієнтів).

In [7]:
qualities_save = [15, 35, 55, 85]
qualities_curve = list(range(5, 100, 5))

reconstructed: dict[int, np.ndarray] = {}
metrics_rows: list[tuple[int, float, float]] = []

for q in qualities_curve:
    rec = compress_decompress_image(img_gray, q)
    mse_v, psnr_v = mse_psnr(img_gray, rec)
    metrics_rows.append((q, mse_v, psnr_v))
    if q in qualities_save:
        reconstructed[q] = rec
        imwrite_unicode(RESULTS_DIR / f"reconstructed_q{q:02d}.png", rec)

for q, mse_v, psnr_v in metrics_rows:
    if q in qualities_save or q % 25 == 0:
        print(f"Q={q:3d}  MSE={mse_v:8.2f}  PSNR={psnr_v:6.2f} дБ")

q_demo = 35
diff = np.abs(img_gray.astype(np.int16) - reconstructed[q_demo].astype(np.int16)).astype(np.uint8)
diff_vis = np.clip(diff * 4, 0, 255).astype(np.uint8)
imwrite_unicode(RESULTS_DIR / "difference_abs_scaled.png", diff_vis)

blk0 = img_gray[:8, :8].astype(np.float64) - 128.0
spec0 = dct2(blk0)
plt.figure(figsize=(4, 3.5))
plt.imshow(np.log1p(np.abs(spec0)), cmap="viridis")
plt.colorbar(label="log(1+|c|)")
plt.title("Спектр DCT першого блоку 8×8")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "dct_spectrum_first_block.png", dpi=120)
plt.close()

Qs, MSEs, PSNRs = zip(*metrics_rows)
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(Qs, PSNRs, "b-o", markersize=4, label="PSNR")
ax1.set_xlabel("Якість Q")
ax1.set_ylabel("PSNR, дБ")
ax1.grid(True, alpha=0.3)
ax2 = ax1.twinx()
ax2.plot(Qs, MSEs, "r-s", markersize=4, label="MSE")
ax2.set_ylabel("MSE")
fig.suptitle("Якість відновлення vs параметр Q")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "psnr_mse_vs_quality.png", dpi=130)
plt.close(fig)

figc, axes = plt.subplots(2, 3, figsize=(11, 7))
panels = [
    ("Оригінал", img_gray),
    ("Відновлення, Q=15", reconstructed[15]),
    ("Відновлення, Q=35", reconstructed[35]),
    ("Відновлення, Q=55", reconstructed[55]),
    ("Відновлення, Q=85", reconstructed[85]),
    ("|різниця|×4 (для Q=35)", diff_vis),
]
for ax, (title, im) in zip(axes.ravel(), panels):
    ax.imshow(im, cmap="gray")
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.suptitle("Порівняння: оригінал, відновлення, різниця")
plt.tight_layout()
figc.savefig(RESULTS_DIR / "comparison_reconstructions.png", dpi=130)
plt.close(figc)

Q= 15  MSE=   78.91  PSNR= 29.16 дБ
Q= 25  MSE=   51.79  PSNR= 30.99 дБ
Q= 35  MSE=   39.22  PSNR= 32.20 дБ
Q= 50  MSE=   30.06  PSNR= 33.35 дБ
Q= 55  MSE=   29.23  PSNR= 33.47 дБ
Q= 75  MSE=    3.60  PSNR= 42.56 дБ
Q= 85  MSE=    2.26  PSNR= 44.59 дБ


## 8. Перевірка збережених файлів

Переконуємося, що всі очікувані результати записані у `RESULTS_DIR`.

In [8]:
expected = [
    "original_gray.png",
    "reconstructed_q15.png",
    "reconstructed_q35.png",
    "reconstructed_q55.png",
    "reconstructed_q85.png",
    "difference_abs_scaled.png",
    "dct_spectrum_first_block.png",
    "psnr_mse_vs_quality.png",
    "comparison_reconstructions.png",
]
missing = [name for name in expected if not (RESULTS_DIR / name).is_file()]
if missing:
    raise FileNotFoundError("Відсутні файли: " + ", ".join(missing))
print("Усі результати лабораторної роботи №5 успішно створено.")

Усі результати лабораторної роботи №5 успішно створено.
